# Example for using PINE

PINE(Pair INterpretation for Entity matching) is an explainable entity matching algorithm.

PINE takes two records(entities) as input, and outputs correlated token pairs as an explanation for an entity-matching decision.

## Install

```bash
git clone https://github.com/foo-bar/pine-explainer.git
pip install .
```

## Advance preparation

### Making your entity matching model in your dataset

PINE can explain arbitrary entity matching models.
You can make your model to analyze by PINE.

In the experiments, we tried `DITTO` model and `py_entitymatching` model in `Magellan Datasets` by using `lemmon` module.
You can see how we make model at [make_DITTO_model_by_lemon.ipynb](../experiments/01_make_model/make_DITTO_model_by_lemon.ipynb) and [make_py_entitymatching_model_by_lemon.ipynb](../experiments/01_make_model/make_py_entitymatching_model_by_lemon.ipynb)

In this example, we use one entity matching model (`py_entitymatching` model for `Structureed Anazon-Google` dataset) in the experiments.

You can download this model from here and set `examples/data/model/magellan/structured_amazon_google` directory.

### Converting your entity matching function to PINE's function

PINE can recognize your entity matching function by converting PINE's function format.

```python
def your_proba_func(
        records_a: pandas.DataFrame,
        records_b: pandas.DataFrame,
        record_id_pairs: pandas.DataFrame,
)->pandas.Series:
   '''Input two entities data, Calc the score.

   Arguments:
        records_a: Entities data in one dataset( index is required) 
        records_b: Entities data in one dataset( index is required) 
        record_id_pairs: Entity pairs between records_a and records_b(index is required)
    Return:
        pandas.Series: socres (-1 <= score <= 1) 
```

`DITTO` function and `py_entitymatching` function have been prepared in PINE (here and here).
In this example use these functions. 

In [ ]:
# Load entity maching function
from pine.matcher.magellan_matcher import make_magellan_matcher_func


model_root_dir = "data/model"
target_dataset_name = "structured_amazon_google"

proba_fn = make_magellan_matcher_func(target_dataset_name, model_root_dir)
#proba_fn = make_transformer_matcher_func(target_dataset_name, model_root_dir)


## Explain py PINE

### Convering your data to PINE's data format

Your data have to be converted to PINE's data format.

- Single row of `pandas.Dataframe` as one your record(one your entity)

In this example, we use lemon dataset which is represented as `pandas.Dataframe`.
`pine-explainer` package contains useful function the lemon dataset.

In [ ]:
from pine.dataset import load_dataset


dataset_root_dir = "data/dataset"
dataset = load_dataset(target_dataset_name, dataset_root_dir)
print("Dataset a")
display(dataset.test.records.a.head())
print("Dataset b")
display(dataset.test.records.b.head())
print("Dataset record-id pairs")
display(dataset.test.record_id_pairs.head())
print("Dataset grand truth labels")
display(dataset.test.labels.head())
print("Test DATA SIZE =", len(dataset.test.record_id_pairs))

In [ ]:
# label = match
match_label_pid = dataset.test.labels[dataset.test.labels == True].index
match_label_record_pair_id = dataset.test.record_id_pairs.loc[match_label_pid]
# label = unmatch
unmatch_label_pid = dataset.test.labels[dataset.test.labels == False].index
unmatch_label_record_pair_id = dataset.test.record_id_pairs.loc[unmatch_label_pid]

### Making explanation

In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModel
from torch.cuda import is_available as cuda_available

from pine.entity import Entity, EntityPair
from pine.explainer import LimeResultPair
from pine.explainer.pine_explainer import PineExplainer

device = "cuda" if cuda_available() else "cpu"

In [ ]:
bert_model_name = "bert-base-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
bert_model = AutoModel.from_pretrained(bert_model_name).to(device)

explainer = PineExplainer(model=bert_model, tokenizer=bert_tokenizer, random_state=0)

def explain_records(
    record_left: pd.DataFrame, record_right: pd.DataFrame, proba_fn, topk=5
):
    # Convert dataset to the format that the matcher function can handle
    entity_left = Entity.from_dataframe(record_left)
    entity_right = Entity.from_dataframe(record_right)
    entity_pair = EntityPair(entity_left, entity_right)
    # Make explanation
    explanation, entity_pair_marged = explainer.make_explanation(entity_pair, proba_fn, topk)
    return explanation, entity_pair_marged


record_left = dataset.test.records.a.loc[[match_label_record_pair_id.iloc[0]["a.rid"]]]
record_right = dataset.test.records.b.loc[[match_label_record_pair_id.iloc[0]["b.rid"]]]
explanation, entity_pair_marged = explain_records(record_left, record_right, proba_fn)

In [ ]:
def display_explanation(explanation: LimeResultPair, entity_pair: EntityPair):
    display("Records")
    display(entity_pair.entity_l.to_dataframe())
    display(entity_pair.entity_r.to_dataframe())
    display("Explanation summary")
    display(f"match_score {explanation.match_score}")
    token_pair_attributions = []
    for i, attr in enumerate(explanation.attributions):
        attribution = {}
        if entity_pair.merged_segment_list[attr.index].segment_list_in_l:
            attribution["token_left"] = entity_pair.entity_l.get_segment_label(
                entity_pair.merged_segment_list[attr.index].segment_list_in_l[0]
            )
        else:
            attribution["token_left"] = ""
        if entity_pair.merged_segment_list[attr.index].segment_list_in_r:
            attribution["token_right"] = entity_pair.entity_r.get_segment_label(
                entity_pair.merged_segment_list[attr.index].segment_list_in_r[0]
            )
        else:
            attribution["token_right"] = ""
        attribution["score"] = attr.score
        token_pair_attributions.append(attribution)
    display(pd.DataFrame(token_pair_attributions))


display_explanation(explanation, entity_pair_marged)

In [ ]:
record_left = dataset.test.records.a.loc[
    [unmatch_label_record_pair_id.iloc[0]["a.rid"]]
]
record_right = dataset.test.records.b.loc[
    [unmatch_label_record_pair_id.iloc[0]["b.rid"]]
]
explanation, entity_pair_marged = explain_records(record_left, record_right, proba_fn)
display_explanation(explanation, entity_pair_marged)